## Section 1 — Install nnU-Net and Dependencies

In [ ]:
"""
CELL 1 — Install nnU-Net v2
Run ONCE. Safe to re-run (idempotent).
"""
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# nnU-Net v2 — the self-configuring segmentation framework
pip("nnunetv2")

# SimpleITK — required by nnU-Net for image I/O
pip("SimpleITK")

# acvl-utils — required by nnU-Net v2
pip("acvl-utils")

# dynamic-network-architectures — required by nnU-Net v2
pip("dynamic-network-architectures")

print("\n✅  Installation complete.")
import nnunetv2
print(f"   nnU-Net version: {nnunetv2.__version__}")

## Section 2 — Setup Environment Variables
nnU-Net requires 3 environment variables pointing to its data directories.
These tell nnU-Net where to find raw data, preprocessed data, and trained models.

In [ ]:
"""
CELL 2 — Configure nnU-Net environment paths

nnU-Net folder structure:
  nnUNet_raw/         ← formatted input data (nnU-Net naming convention)
  nnUNet_preprocessed/ ← nnU-Net preprocessed (auto-generated)
  nnUNet_results/     ← trained model weights & checkpoints
"""
import os
from pathlib import Path

BASE_DIR   = Path("/home/moamed/canada_me/explainable_diseas/implementation")
PROCESSED  = Path("/media/moamed/Data/yale-processed")    # our preprocessed NIfTIs
NNUNET_DIR = Path("/media/moamed/Data/nnunet")             # nnU-Net working directory

# Create nnU-Net directory structure
NNUNET_RAW          = NNUNET_DIR / "nnUNet_raw"
NNUNET_PREPROCESSED = NNUNET_DIR / "nnUNet_preprocessed"
NNUNET_RESULTS      = NNUNET_DIR / "nnUNet_results"

for d in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

# Set environment variables (required by all nnU-Net commands)
os.environ["nnUNet_raw"]          = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"]      = str(NNUNET_RESULTS)

# Paths for our specific dataset
MANIFEST    = BASE_DIR / "outputs" / "processed_manifest.csv"
SPLITS_JSON = BASE_DIR / "outputs" / "splits.json"
LOG_DIR     = BASE_DIR / "outputs" / "segmentation_log"
LOG_DIR.mkdir(exist_ok=True)
SEG_LOG_CSV = LOG_DIR / "segmentation_log.csv"

# nnU-Net dataset ID for our Yale Brain Mets data
# We use Dataset500 (500-999 is reserved for user datasets)
DATASET_ID   = 500
DATASET_NAME = f"Dataset{DATASET_ID:03d}_YaleBrainMets"
YALE_RAW_DIR = NNUNET_RAW / DATASET_NAME

print("nnU-Net directories:")
print(f"  Raw          : {NNUNET_RAW}")
print(f"  Preprocessed : {NNUNET_PREPROCESSED}")
print(f"  Results      : {NNUNET_RESULTS}")
print(f"  Dataset      : {DATASET_NAME}")
print()
print("Env vars set ✅")

## Section 3 — Download BraTS Pretrained Weights

We use nnU-Net's **Dataset137_BraTS2021** pretrained weights.
These were trained on the BraTS 2021 challenge dataset (1,251 glioblastoma cases)
and achieve Dice ~0.908 on brain tumors.

**Why BraTS weights work for Yale Brain Mets**:
- Same 4 modalities: T1, T1ce(POST), T2, FLAIR
- Both are brain MRI with contrast-enhancing tumors
- nnU-Net normalizes internally → scanner differences handled automatically
- Brain mets show same enhancement pattern as glioblastoma (ET/TC/ED regions)

Download from the official nnU-Net zenodo: https://zenodo.org/record/7786688

In [ ]:
"""
CELL 3 — Download BraTS 2021 pretrained weights (Dataset137)

nnU-Net provides pretrained weights via the nnUNetv2_download_pretrained_model_by_url command.
Alternatively download manually from Zenodo and install with:
  nnUNetv2_install_pretrained_model_from_zip <path_to_zip>

The weights folder will be at:
  $nnUNet_results/Dataset137_BraTS2021/nnUNetTrainer__nnUNetPlans__3d_fullres/
"""
import subprocess

BRATS_MODEL_DIR = NNUNET_RESULTS / "Dataset137_BraTS2021" / "nnUNetTrainer__nnUNetPlans__3d_fullres"

if BRATS_MODEL_DIR.exists() and any(BRATS_MODEL_DIR.glob("fold_*")):
    print(f"✅  BraTS model already present: {BRATS_MODEL_DIR}")
    folds = sorted([f.name for f in BRATS_MODEL_DIR.glob("fold_*")])
    print(f"   Folds: {folds}")
else:
    print("Downloading BraTS 2021 pretrained weights...")
    print("This is ~1.5 GB and may take several minutes.")
    print()
    
    # nnU-Net v2 download command
    result = subprocess.run(
        ["nnUNetv2_download_pretrained_model_by_url",
         "https://zenodo.org/record/7786688/files/Dataset137_BraTS2021.zip"],
        capture_output=True, text=True, env={**os.environ}
    )
    if result.returncode == 0:
        print("✅  Download and installation complete.")
    else:
        print("❌  Download failed. Manual install instructions:")
        print("  1. Download from: https://zenodo.org/record/7786688")
        print("     File: Dataset137_BraTS2021.zip")
        print(f"  2. Run: nnUNetv2_install_pretrained_model_from_zip <path_to_zip>")
        print()
        print("STDERR:", result.stderr[:500])

## Section 4 — Prepare Input Files for nnU-Net

nnU-Net requires a **strict naming convention** for input files:
```
{CaseID}_{ModalityIndex:04d}.nii.gz
```

For BraTS (4 modalities):
| Index | Modality | Our column |
|-------|----------|------------|
| 0000 | T1 | path_PRE |
| 0001 | T1ce / POST | path_POST |
| 0002 | T2 | path_T2 |
| 0003 | FLAIR | path_FLAIR |

We create **symlinks** (not copies) to avoid duplicating 19 GB of data.

In [ ]:
"""
CELL 4 — Prepare nnU-Net input folder with symlinks

Creates: $nnUNet_raw/Dataset500_YaleBrainMets/imagesTs/
   YG_01M98EKKAR50_2016-11-13_0000.nii.gz  → path_PRE
   YG_01M98EKKAR50_2016-11-13_0001.nii.gz  → path_POST
   YG_01M98EKKAR50_2016-11-13_0002.nii.gz  → path_T2
   YG_01M98EKKAR50_2016-11-13_0003.nii.gz  → path_FLAIR
"""
import pandas as pd
import csv

# ── Create nnU-Net folder structure ───────────────────────────────────────────
IMAGES_TS = YALE_RAW_DIR / "imagesTs"   # Ts = test/inference (no labels needed)
SEG_OUT   = NNUNET_DIR / "segmentations" / "yale_predictions"
IMAGES_TS.mkdir(parents=True, exist_ok=True)
SEG_OUT.mkdir(parents=True, exist_ok=True)

# ── Modality index mapping (matches BraTS Dataset137 channel order) ───────────
MOD_MAP = {
    "path_PRE" : "0000",   # T1
    "path_POST": "0001",   # T1ce / contrast-enhanced POST
    "path_T2"  : "0002",   # T2
    "path_FLAIR": "0003",  # FLAIR
}

df = pd.read_csv(MANIFEST)
# Only process COMPLETE visits (all 4 modalities present)
df_complete = df[df["complete"] == True].reset_index(drop=True)
print(f"Complete visits to process: {len(df_complete)}")

created   = 0
skipped   = 0
missing   = 0
case_ids  = []   # track all case IDs for inference

for _, row in df_complete.iterrows():
    # CaseID = PatientID_VisitDate  (no slashes, nnU-Net-safe)
    case_id = f"{row['patient_id']}_{row['visit_date']}"
    case_ids.append(case_id)
    
    for col, idx in MOD_MAP.items():
        src = Path(str(row[col]))
        dst = IMAGES_TS / f"{case_id}_{idx}.nii.gz"
        
        if dst.exists():
            skipped += 1
            continue
        if not src.exists():
            print(f"  ⚠️  Missing: {src}")
            missing += 1
            continue
        
        # Symlink — no data duplication
        dst.symlink_to(src)
        created += 1

print(f"\nSymlinks created : {created}")
print(f"Already existed  : {skipped}")
print(f"Missing files    : {missing}")
print(f"Total cases      : {len(case_ids)}")
print(f"\nInput folder: {IMAGES_TS}")
print(f"Output folder: {SEG_OUT}")

## Section 5 — Run nnU-Net Inference

Runs `nnUNetv2_predict` using the BraTS 2021 pretrained weights.

**Estimated time**: ~2 min/case on GPU → 889 cases ≈ **30 hours** total.  
Use `TIME_CAP_HOURS` to stop cleanly — it's fully **idempotent** (already-done cases skip).

**Memory**: ~6 GB VRAM (3d_fullres). If OOM, use `3d_lowres` (faster, slightly less accurate).

In [ ]:
"""
CELL 5 — Run nnU-Net inference (batch with time cap)
"""
import time
import csv as csv_mod
from datetime import datetime
from tqdm.auto import tqdm

# ── Settings ──────────────────────────────────────────────────────────────────
TIME_CAP_HOURS = 5.0          # ← set to None to run until complete
CONFIGURATION  = "3d_fullres" # 3d_fullres (best) | 3d_lowres (faster, less VRAM)
FOLDS          = (0, 1, 2, 3, 4)   # all 5 folds = best accuracy via ensemble
USE_TTA        = True         # Test-time augmentation (mirroring) — improves Dice ~1%
DEVICE         = "cuda"       # cuda | cpu | mps

# ── Import nnU-Net predictor ──────────────────────────────────────────────────
import torch
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from nnunetv2.imageio.simpleitk_reader_writer import SimpleITKIO
from batchgenerators.utilities.file_and_folder_operations import join

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Model folder (BraTS 2021 pretrained weights) ──────────────────────────────
MODEL_FOLDER = str(NNUNET_RESULTS / "Dataset137_BraTS2021" /
                   f"nnUNetTrainer__nnUNetPlans__{CONFIGURATION}")
print(f"\nModel: {MODEL_FOLDER}")
assert Path(MODEL_FOLDER).exists(), (
    f"Model not found! Run Section 3 first to download BraTS weights.\n"
    f"Expected: {MODEL_FOLDER}"
)

# ── Find cases that still need segmentation ───────────────────────────────────
all_cases = sorted(set(
    f.stem.rsplit("_", 1)[0]   # strip _0000 suffix → case ID
    for f in IMAGES_TS.glob("*_0000.nii.gz")
))
pending = [
    c for c in all_cases
    if not (SEG_OUT / f"{c}.nii.gz").exists()
]
print(f"\nTotal cases    : {len(all_cases)}")
print(f"Already done   : {len(all_cases) - len(pending)}")
print(f"Pending        : {len(pending)}")

if len(pending) == 0:
    print("\n✅  All cases already segmented!")
else:
    # ── Initialise predictor ONCE (loads model weights to GPU) ────────────────
    print("\nLoading model weights to GPU...")
    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=USE_TTA,
        perform_everything_on_device=True,
        device=torch.device(DEVICE),
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        MODEL_FOLDER,
        use_folds=FOLDS,
        checkpoint_name="checkpoint_final.pth",
    )
    print("Model loaded ✅")
    
    # ── Log helper ────────────────────────────────────────────────────────────
    def log_seg(case_id, status, time_s=0.0, error_msg=""):
        write_header = not SEG_LOG_CSV.exists()
        with open(SEG_LOG_CSV, "a", newline="") as f:
            w = csv_mod.writer(f)
            if write_header:
                w.writerow(["case_id", "status", "time_s", "error_msg", "timestamp"])
            w.writerow([case_id, status, round(time_s, 2), error_msg,
                        datetime.now().strftime("%Y-%m-%d %H:%M:%S")])
    
    # ── Batch inference loop ──────────────────────────────────────────────────
    rw       = SimpleITKIO()
    results  = {"success": 0, "failed": 0}
    t_batch  = time.time()
    cap_s    = TIME_CAP_HOURS * 3600 if TIME_CAP_HOURS else float("inf")
    
    print(f"\nRunning inference ({len(pending)} cases, cap={TIME_CAP_HOURS}h)...")
    
    with tqdm(total=len(pending), desc="nnU-Net segmentation", unit="case") as pbar:
        for case_id in pending:
            
            if time.time() - t_batch >= cap_s:
                print(f"\n⏰  Time cap reached ({TIME_CAP_HOURS}h). Re-run to continue.")
                break
            
            t0 = time.time()
            try:
                # Build list of 4 input files for this case
                input_files = [
                    str(IMAGES_TS / f"{case_id}_0000.nii.gz"),  # T1 / PRE
                    str(IMAGES_TS / f"{case_id}_0001.nii.gz"),  # T1ce / POST
                    str(IMAGES_TS / f"{case_id}_0002.nii.gz"),  # T2
                    str(IMAGES_TS / f"{case_id}_0003.nii.gz"),  # FLAIR
                ]
                output_file = str(SEG_OUT / case_id)  # nnU-Net appends .nii.gz
                
                # Load images
                img, props = rw.read_images(input_files)
                
                # Run prediction (returns numpy array)
                seg = predictor.predict_single_npy_array(
                    img, props, 
                    segmentation_previous_stage=None,
                    output_file_truncated=output_file,
                    save_or_return_probabilities=False,
                )
                
                elapsed = time.time() - t0
                results["success"] += 1
                log_seg(case_id, "success", elapsed)
                
            except Exception as e:
                elapsed = time.time() - t0
                results["failed"] += 1
                log_seg(case_id, "failed", elapsed, str(e)[:200])
                print(f"\n  ❌ {case_id}: {e}")
            
            wall_left = max(0, cap_s - (time.time() - t_batch))
            pbar.set_postfix({**results, "cap_left": f"{wall_left/60:.0f}m"})
            pbar.update(1)
    
    total_h = (time.time() - t_batch) / 3600
    print(f"\nBatch done in {total_h:.2f}h")
    print(f"  Success : {results['success']}")
    print(f"  Failed  : {results['failed']}")

## Section 6 — Inspect Segmentation Results

In [ ]:
"""
CELL 6 — Check progress and visualise example segmentations
"""
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

seg_files = sorted(SEG_OUT.glob("*.nii.gz"))
print(f"Segmentations done: {len(seg_files)} / {len(all_cases)}")

if SEG_LOG_CSV.exists():
    df_log = pd.read_csv(SEG_LOG_CSV)
    print(f"Log entries: {len(df_log)}")
    print(df_log["status"].value_counts().to_string())
    if "time_s" in df_log.columns:
        done = df_log[df_log["status"] == "success"]
        if len(done) > 0:
            avg = done["time_s"].mean()
            remaining = len(all_cases) - len(seg_files)
            print(f"\nAvg time/case : {avg:.1f}s")
            print(f"ETA remaining : ~{avg * remaining / 3600:.1f}h")

if len(seg_files) == 0:
    print("\n⚠️  No segmentations yet — run Section 5 first.")
else:
    # ── Visualise one example ─────────────────────────────────────────────────
    seg_path = seg_files[0]
    case_id  = seg_path.name.replace(".nii.gz", "")
    
    seg_arr  = nib.load(str(seg_path)).get_fdata(dtype=np.float32).astype(np.int8)
    post_path = IMAGES_TS / f"{case_id}_0001.nii.gz"
    post_arr  = nib.load(str(post_path)).get_fdata(dtype=np.float32) if post_path.exists() else None
    
    # Find slice with most tumor
    tumor_mask  = seg_arr > 0
    best_slice  = int(np.argmax(tumor_mask.sum(axis=(0, 1))))
    
    label_names = {0: "Background", 1: "Edema (ED)", 2: "Necrotic core (NCR)", 3: "Enhancing tumor (ET)"}
    label_colors = {0: "black", 1: "yellow", 2: "blue", 3: "red"}
    cmap = plt.matplotlib.colors.ListedColormap(["black", "yellow", "blue", "red"])
    
    n_cols = 3 if post_arr is not None else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
    fig.suptitle(f"{case_id} | slice {best_slice}", fontsize=10, fontweight="bold")
    
    axes[0].imshow(seg_arr[:, :, best_slice].T, cmap=cmap, vmin=0, vmax=3, origin="lower")
    axes[0].set_title("Segmentation mask")
    patches = [mpatches.Patch(color=label_colors[k], label=v) for k, v in label_names.items() if k > 0]
    axes[0].legend(handles=patches, loc="lower right", fontsize=7)
    axes[0].axis("off")
    
    if post_arr is not None:
        axes[1].imshow(post_arr[:, :, best_slice].T, cmap="gray", origin="lower")
        axes[1].set_title("POST (T1ce)")
        axes[1].axis("off")
        
        axes[2].imshow(post_arr[:, :, best_slice].T, cmap="gray", origin="lower")
        overlay = np.ma.masked_where(seg_arr[:, :, best_slice] == 0, seg_arr[:, :, best_slice])
        axes[2].imshow(overlay.T, cmap=cmap, vmin=0, vmax=3, alpha=0.5, origin="lower")
        axes[2].set_title("POST + mask overlay")
        axes[2].axis("off")
    
    plt.tight_layout()
    plt.savefig(LOG_DIR / "example_segmentation.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    # Label statistics
    print(f"\nLabel statistics for {case_id}:")
    for label, name in label_names.items():
        count = (seg_arr == label).sum()
        pct   = 100 * count / seg_arr.size
        print(f"  {label} {name:25s}: {count:8,} voxels ({pct:.2f}%)")

## Section 7 — Update Manifest with Segmentation Paths

Adds a `path_seg` column to `processed_manifest.csv` pointing to each visit's
tumor segmentation mask. This is what the CNN notebook uses to get **real labels**
instead of pseudo-labels based on signal intensity.

In [ ]:
"""
CELL 7 — Add path_seg column to processed_manifest.csv

Also computes per-visit tumor metrics from the segmentation:
  - et_volume_mm3   : Enhancing Tumor volume (mm³)
  - tc_volume_mm3   : Tumor Core volume (mm³)
  - wt_volume_mm3   : Whole Tumor volume (mm³)
These become the REAL progression labels (replace pseudo-labels from Cell 4).
"""
import pandas as pd
import nibabel as nib
import numpy as np
from tqdm.auto import tqdm

df = pd.read_csv(MANIFEST)
print(f"Manifest: {len(df)} rows")

seg_records = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Computing tumor volumes"):
    case_id  = f"{row['patient_id']}_{row['visit_date']}"
    seg_path = SEG_OUT / f"{case_id}.nii.gz"
    
    rec = {"case_id": case_id, "path_seg": None,
           "et_volume_mm3": np.nan, "tc_volume_mm3": np.nan, "wt_volume_mm3": np.nan}
    
    if seg_path.exists():
        rec["path_seg"] = str(seg_path)
        img  = nib.load(str(seg_path))
        seg  = img.get_fdata(dtype=np.float32).astype(np.int8)
        # Voxel volume in mm³ from header spacing
        vox_vol = float(np.prod(img.header.get_zooms()[:3]))
        rec["et_volume_mm3"] = round(float((seg == 3).sum()) * vox_vol, 1)
        rec["tc_volume_mm3"] = round(float((seg >= 2).sum()) * vox_vol, 1)
        rec["wt_volume_mm3"] = round(float((seg >= 1).sum()) * vox_vol, 1)
    
    seg_records.append(rec)

df_seg = pd.DataFrame(seg_records)
df_seg["patient_id"] = df_seg["case_id"].str.rsplit("_", n=1).str[0]
df_seg["visit_date"] = df_seg["case_id"].str.rsplit("_", n=1).str[1]

# Merge back into manifest
df_updated = df.merge(
    df_seg[["patient_id", "visit_date", "path_seg",
            "et_volume_mm3", "tc_volume_mm3", "wt_volume_mm3"]],
    on=["patient_id", "visit_date"], how="left"
)

df_updated.to_csv(MANIFEST, index=False)
n_seg = df_updated["path_seg"].notna().sum()
print(f"\n✅  Manifest updated: {MANIFEST}")
print(f"   Visits with segmentation : {n_seg} / {len(df_updated)}")
print(f"   Visits without seg       : {len(df_updated) - n_seg}")
print()
print("Tumor volume statistics (segmented visits):")
seg_done = df_updated[df_updated["path_seg"].notna()]
for col in ["et_volume_mm3", "tc_volume_mm3", "wt_volume_mm3"]:
    print(f"  {col:20s}: mean={seg_done[col].mean():.0f}, "
          f"median={seg_done[col].median():.0f}, "
          f"max={seg_done[col].max():.0f} mm³")

## Section 8 — Build Real Labels from Tumor Volumes

Replace pseudo-labels (POST signal ratio) with **real labels based on ET volume change**:
- `label = 1` (progressive) if Enhancing Tumor volume increased **>20%** vs previous visit
- `label = 0` (stable/regressing) otherwise

This is the standard RANO (Response Assessment in Neuro-Oncology) approach:
bidimensional diameter change, but we use volumetric ET change as a proxy.

In [ ]:
"""
CELL 8 — Build real progression labels from ET volume change
Saves updated manifest with 'label_seg' column.
Use this in place of the pseudo-label 'label' from notebook 04.
"""
import pandas as pd
import numpy as np

df = pd.read_csv(MANIFEST)

# Only patients with at least 2 segmented visits
df_seg = df[df["path_seg"].notna()].copy()

PROGRESSION_THRESHOLD = 0.20   # >20% ET volume increase = progressive (RANO-inspired)

records        = []
skipped_single = 0
skipped_no_seg = 0

for pid, grp in df_seg.groupby("patient_id"):
    grp = grp.sort_values("visit_date").reset_index(drop=True)
    if len(grp) < 2:
        skipped_single += 1
        continue
    
    et_vols = grp["et_volume_mm3"].values
    valid   = [v for v in et_vols if not np.isnan(v)]
    if len(valid) < 2:
        skipped_no_seg += 1
        continue
    
    labels     = [0] * len(grp)
    prev_valid = None
    for i, vol in enumerate(et_vols):
        if np.isnan(vol):
            labels[i] = labels[i - 1] if i > 0 else 0
            continue
        if prev_valid is not None and prev_valid > 0:
            ratio      = vol / (prev_valid + 1e-6)
            labels[i]  = int(ratio > (1 + PROGRESSION_THRESHOLD))
        prev_valid = vol
    labels[0] = labels[1] if len(labels) > 1 else 0
    
    grp_copy = grp.copy()
    grp_copy["label_seg"] = labels
    records.append(grp_copy)

if records:
    df_labelled = pd.concat(records, ignore_index=True)
    # Merge label_seg back into full manifest
    df = df.merge(
        df_labelled[["patient_id", "visit_date", "label_seg"]],
        on=["patient_id", "visit_date"], how="left"
    )
    df.to_csv(MANIFEST, index=False)
    
    dist = df_labelled["label_seg"].value_counts().to_dict()
    print(f"Real labels built from ET volume change (>{PROGRESSION_THRESHOLD*100:.0f}% increase):")
    print(f"  Labelled visits   : {len(df_labelled)}")
    print(f"  Skipped (1 visit) : {skipped_single}")
    print(f"  Skipped (no seg)  : {skipped_no_seg}")
    print(f"  Label distribution: {dist}")
    print(f"  Progressive rate  : {dist.get(1, 0) / len(df_labelled) * 100:.1f}%")
    print(f"\n✅  Manifest updated with 'label_seg' column: {MANIFEST}")
else:
    print("⚠️  No labelled visits — run Section 5 first to generate segmentations.")

## Section 9 — Upload Updated Manifest to Kaggle

Uploads the new `processed_manifest.csv` (now with `path_seg`, `et_volume_mm3`,
and `label_seg` columns) to Kaggle as a new version of the existing dataset.

In [ ]:
"""
CELL 9 — Upload updated manifest to Kaggle

Uploads only the manifest CSV (tiny file) as a new VERSION of the existing
Kaggle dataset 'mohamedmohamed23/yale-processed-manifest'.

This does NOT re-upload the NIfTI files.
"""
import subprocess, shutil, json
from pathlib import Path

# ── Stage folder for Kaggle upload ────────────────────────────────────────────
STAGE_DIR = BASE_DIR / "outputs" / "manifest_kaggle_stage"
STAGE_DIR.mkdir(exist_ok=True)

# Copy manifest to stage folder
shutil.copy(MANIFEST, STAGE_DIR / "processed_manifest.csv")

# Create dataset-metadata.json for the manifest dataset
metadata = {
    "title": "Yale Brain Mets — Processed Manifest",
    "id": "mohamedmohamed23/yale-processed-manifest",
    "licenses": [{"name": "CC0-1.0"}]
}
with open(STAGE_DIR / "dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Show what's being uploaded
import pandas as pd
df_check = pd.read_csv(STAGE_DIR / "processed_manifest.csv")
print(f"Manifest to upload:")
print(f"  Rows    : {len(df_check)}")
print(f"  Columns : {df_check.columns.tolist()}")
has_seg    = df_check['path_seg'].notna().sum() if 'path_seg' in df_check.columns else 0
has_labels = df_check['label_seg'].notna().sum() if 'label_seg' in df_check.columns else 0
print(f"  With segmentation path  : {has_seg}")
print(f"  With real label_seg     : {has_labels}")
print()

# ── Upload as a new VERSION (not create — dataset already exists) ─────────────
print("Uploading manifest to Kaggle...")
result = subprocess.run(
    ["kaggle", "datasets", "version",
     "-p", str(STAGE_DIR),
     "-m", "Add path_seg, et_volume_mm3, tc_volume_mm3, wt_volume_mm3, label_seg columns from nnUNet segmentation",
     "--dir-mode", "zip"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅  Manifest uploaded successfully!")
    print(result.stdout)
else:
    print("❌  Upload failed:")
    print(result.stderr)